In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

# Ridership Analysis - 2019 to 2026

In [2]:
ridership_2017_2019 = pd.read_parquet("ridership_2017_2019_temporal.parquet")
ridership_2020_2024 = pd.read_parquet("ridership_2020_2024_temporal.parquet")
ridership_2025_2026 = pd.read_parquet("ridership_2025_2026_temporal.parquet")

ridership_2017_2019_station = pd.read_parquet("ridership_2017_2019_station.parquet")
ridership_2020_2024_station = pd.read_parquet("ridership_2020_2024_station.parquet")
ridership_2025_2026_station = pd.read_parquet("ridership_2025_2026_station.parquet")

station_complexes = pd.read_parquet("station_complexes.parquet")

od_2021_september = pd.read_parquet("od_2021_september.parquet")
od_2025_september = pd.read_parquet("od_2025_september.parquet")
od_2026_september = pd.read_parquet("od_2026_september.parquet")

## Systemwide Trends

In [3]:
combined_ridership = pd.concat(
    [ridership_2017_2019, ridership_2020_2024, ridership_2025_2026],
    ignore_index=True
)

combined_ridership["date"] = pd.to_datetime(combined_ridership["date"])
combined_ridership = combined_ridership.sort_values(
    ["date", "time_period"]
).reset_index(drop=True)

### Total Ridership

In [4]:
annual_ridership = (
    combined_ridership.assign(year=combined_ridership["date"].dt.year)
    .groupby("year", as_index=False)["total_ridership"]
    .sum()
)

annual_ridership_fig = px.bar(
    annual_ridership,
    x="year",
    y="total_ridership",
    title="Total Ridership per Year",
    labels={"year": "Year", "total_ridership": "Total Ridership"},
)

max_date = combined_ridership["date"].max()

annual_ridership_fig.update_xaxes(
    tickmode="array",
    tickvals=annual_ridership["year"],
    ticktext=[
        f"{year}*" if year == max_date.year else str(year)
        for year in annual_ridership["year"]
    ],
)

annual_ridership_fig.add_annotation(
    x=0.5,
    y=-0.2,
    xref="paper",
    yref="paper",
    text=f"* 2026 data through {max_date:%b %d, %Y} (the maximum date in the dataset).",
    showarrow=False,
    xanchor="center",
)
annual_ridership_fig.update_layout(margin=dict(b=90))

annual_ridership_fig.show()

In [5]:
daily_totals = combined_ridership.groupby("date")["total_ridership"].sum()
baseline_2019 = daily_totals.loc[daily_totals.index.year == 2019].mean() * 7

weekly_total = (
    daily_totals.resample("W-SUN")
    .sum()
    .rename("weekly_ridership")
    .reset_index()
)
weekly_total = weekly_total.loc[weekly_total["date"] >= "2019-01-01"].reset_index(drop=True)
weekly_total["percent_change"] = (
    weekly_total["weekly_ridership"] / baseline_2019 - 1
) * 100

weekly_ridership_fig = px.line(
    weekly_total,
    x="date",
    y="percent_change",
    title="Weekly Ridership vs. Average Week in 2019",
    labels={"date": "Week ending", "percent_change": "Change (%)"},
)
weekly_ridership_fig.add_hline(y=0, line_dash="dash", line_color="gray")

last_full_week = max_date.normalize() - pd.Timedelta(
    days=(max_date.weekday() + 1) % 7
)
weekly_ridership_fig.update_xaxes(
    range=[weekly_total["date"].min(), last_full_week]
)
weekly_ridership_fig.add_annotation(
    x=0.5,
    y=-0.2,
    xref="paper",
    yref="paper",
    text=(
        f"Baseline: 2019 average daily ridership × 7. Data through {max_date:%b %d, %Y}; "    ),
    showarrow=False,
    xanchor="center",
)
weekly_ridership_fig.update_layout(margin=dict(b=90))
weekly_ridership_fig.show()

In [6]:
jan_sep_totals = (
    combined_ridership.loc[combined_ridership["date"].dt.month <= 9]
    .assign(year=lambda df: df["date"].dt.year)
    .groupby("year")["total_ridership"]
    .sum()
)

comparisons = [(2019, 2026), (2025, 2026)]

jan_sep_comparison = pd.DataFrame(
    [
        {
            "year comparison": f"{base_year}-{compare_year}",
            "percent diff": (
                jan_sep_totals[compare_year] / jan_sep_totals[base_year] - 1
            ) * 100,
            "total ridership diff": (
                jan_sep_totals[compare_year] - jan_sep_totals[base_year]
            ),
        }
        for base_year, compare_year in comparisons
    ]
)

jan_sep_comparison.style.format({
    "percent diff": "{:.2f}%",
    "total ridership diff": "{:,.0f}",
})

,year comparison,percent diff,total ridership diff
0,2019-2026,-21.78%,"-275,496,399"
1,2025-2026,2.51%,"24,178,815"


#### Takeaway
Total ridership has not recovered to pre-COVID 2019 numbers. Total ridership from January - September 2026 is 21.78% lower than total ridership from January - September 2019. We are seeing year over year increases in total ridership since 2020; ridership from January - September 2026 is 2.51% higher than ridership from January - September 2025. 

### Weekday vs Weekend Ridership

In [7]:
daily_ridership_by_date = (
    combined_ridership
    .groupby(["date", "day_of_week"], as_index=False)["total_ridership"]
    .sum()
    .assign(
        year=lambda df: df["date"].dt.year.astype(str),
        day_type=lambda df: df["day_of_week"].map(
            lambda day: "Weekend" if day in ["Saturday", "Sunday"] else "Weekday"
        ),
    )
)


average_daily_ridership_by_day_type_year = (
    daily_ridership_by_date
    .groupby(["day_type", "year"], as_index=False)["total_ridership"]
    .median()
    .rename(columns={"total_ridership": "average_daily_ridership"})
)


average_daily_ridership_fig = px.bar(
    average_daily_ridership_by_day_type_year,
    x="day_type",
    y="average_daily_ridership",
    color="year",
    text=average_daily_ridership_by_day_type_year["year"].astype(str),
    barmode="group",
    category_orders={"day_type": ["Weekday", "Weekend"]},
    title="Average Total Ridership per Weekday and Weekend Day by Year",
    labels={
        "day_type": "Day Type",
        "average_daily_ridership": "Average Total Ridership per Day",
        "year": "Year",
    },
)

average_daily_ridership_fig.update_traces(textposition="outside")
average_daily_ridership_fig.update_layout(showlegend=False)
average_daily_ridership_fig.show()

In [8]:
weekday_weekend_totals = (
    daily_ridership_by_date.loc[
        daily_ridership_by_date["date"].dt.month.le(9)
    ]
    .assign(year=lambda df: df["date"].dt.year.astype(str))
    .groupby(["year", "day_type"])["total_ridership"]
    .sum()
)

def make_weekday_weekend_comparison(base_year, compare_year):
    rows = []
    for day_type in ["Weekday", "Weekend"]:
        base_total = weekday_weekend_totals.loc[(str(base_year), day_type)]
        compare_total = weekday_weekend_totals.loc[(str(compare_year), day_type)]
        rows.append({
            "day type": day_type,
            "percent change": (compare_total / base_total - 1) * 100,
        })
    return pd.DataFrame(rows)

table_2019_2026_weekday_weekend = make_weekday_weekend_comparison(2019, 2026)
table_2025_2026_weekday_weekend = make_weekday_weekend_comparison(2025, 2026)

#### Takeaway

The average weekday and average weekend ridership trends since 2019 show that ridership on weekdays so far in 2026 have had the most drastic change from 2019. The average weekday ridership from 2026 (data from January - September) is 24.47% lower than average weekday ridership from 2019, while weekend ridership is only 8.28% lower when comparing the same time ranges.

When looking at how ridership has changed within the last year, average weekday and weekend ridership is actually changing at about the same rate, with the average weekday ridership from 2026 (January - September) being 2.51% higher than the average weekday ridership from 2019. average weekend ridership increased simiarly, at 2.50% higher in 2026 than 2025. 

This indicates while weekday ridership is recovering recently at approximately the same rate that weekend ridership is recovering, there is still a larger gap between weekday ridership in 2019 and weekday ridership in 2026 that has not yet been regained.

## Temporal Changes

### Service Time Period
Analyzing ridership by service time period: 
- Weekend Late night: midnight to 6:30 AM, weekend
- Weekday Late night: midnight to 6:30 AM, weekday
- Weekend Daytime/Evening: 6:30 AM to midnight, weekend
- AM Rush Hour: 6:30-9:30 AM, weekday
- PM Rush Hour: 3:30 - 8 PM, weekday
- Evening:  8 PM to midnight, weekday
- Midday: 9:30 AM to 3:30 PM, weekday

In [9]:
jan_sep_by_period_year = (
    combined_ridership.loc[combined_ridership["date"].dt.month <= 9]
    .assign(year=lambda df: df["date"].dt.year.astype(str))
    .groupby(["year", "time_period"])["total_ridership"]
    .sum()
)

jan_sep_totals_by_year = (
    combined_ridership.loc[combined_ridership["date"].dt.month <= 9]
    .assign(year=lambda df: df["date"].dt.year.astype(str))
    .groupby("year")["total_ridership"]
    .sum()
)

time_periods = combined_ridership["time_period"].dropna().unique()

def make_time_period_comparison_table(base_year, compare_year):
    base_year, compare_year = str(base_year), str(compare_year)

    rows = []
    categories = ["total", *time_periods]

    for category in categories:
        if category == "total":
            base_value = jan_sep_totals_by_year.loc[base_year]
            compare_value = jan_sep_totals_by_year.loc[compare_year]
        else:
            base_value = jan_sep_by_period_year.loc[(base_year, category)]
            compare_value = jan_sep_by_period_year.loc[(compare_year, category)]

        rows.append({
            "total/time period": category,
            "percent change": (compare_value / base_value - 1) * 100,
            "total ridership change": compare_value - base_value,
        })

    return pd.DataFrame(rows).sort_values(by="percent change", key=abs, ascending=False)

table_2019_2026_time_period = make_time_period_comparison_table(2019, 2026)
table_2025_2026_time_period = make_time_period_comparison_table(2025, 2026)

In [10]:
chart_data = jan_sep_by_period_year.rename("total_ridership").reset_index()
chart_data = chart_data.loc[chart_data["year"].isin(["2019", "2025", "2026"])].copy()

period_order = (
    chart_data.loc[chart_data["year"].eq("2019")]
    .sort_values("total_ridership", ascending=False)["time_period"]
    .tolist()
)

fig = px.bar(
    chart_data,
    x="time_period",
    y="total_ridership",
    color="year",
    barmode="group",
    category_orders={
        "time_period": period_order,
        "year": ["2019", "2025", "2026"],
    },
    title="January–September Ridership by Time Period",
    labels={
        "time_period": "Time Period",
        "total_ridership": "Total Ridership",
        "year": "Year",
    },
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

In [11]:
def plot_time_period_changes(table, comparison):
    chart_data = table.copy()
    chart_data["direction"] = chart_data["percent change"].apply(
        lambda value: "Increase" if value >= 0 else "Decrease"
    )
    chart_data = chart_data.sort_values(
        "percent change", key=lambda values: values.abs()
    )

    percent_fig = px.bar(
        chart_data,
        x="percent change",
        y="total/time period",
        orientation="h",
        color="direction",
        color_discrete_map={"Increase": "#00cc96", "Decrease": "#EF553B"},
        hover_data={"total ridership change": ":,.0f"},
        title=f"Ridership Change by Time Period: {comparison} (%)",
        labels={
            "percent change": "Change (%)",
            "total/time period": "Time Period",
            "total ridership change": "Ridership change",
        },
    )
    percent_fig.add_vline(x=0, line_dash="dash", line_color="gray")
    percent_fig.update_layout(showlegend=False)
    percent_fig.show()


plot_time_period_changes(table_2019_2026_time_period, "2019–2026")
plot_time_period_changes(table_2025_2026_time_period, "2025–2026")




#### Takeaway

For total ridership between Jan - Sept 2019 and Jan - Sept 2026, ridership during AM Rush Hours (decreased by 30.8%) and PM Rush Hours (decreased by 25%) showed the largest change.

For total ridership between Jan - Sept 2025 and Jan - Sept 2026, ridership during weekend late night (increased by 7.4%) and weekday evening service (increased by 5.2%) showed the largest change. 

The highest ridership weekday time periods, AM and PM Rush Hour, are not recovering as quickly as other time periods, which is a key factor for why overall weekday ridership still remains much lower than 2019 numbers.


### Day of the Week

In [12]:
jan_sep_daily = daily_ridership_by_date.loc[
    daily_ridership_by_date["date"].dt.month <= 9
]

jan_sep_by_year_type = (
    jan_sep_daily.groupby(["year", "day_type"])["total_ridership"].sum()
)
jan_sep_by_year = jan_sep_daily.groupby("year")["total_ridership"].sum()

jan_sep_by_day_year = (
    jan_sep_daily.groupby(["year", "day_of_week"])["total_ridership"].sum()
)
day_of_week_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]

def make_day_of_week_comparison_table(base_year, compare_year):
    base_year, compare_year = str(base_year), str(compare_year)
    rows = []

    base_total = jan_sep_by_year.loc[base_year]
    compare_total = jan_sep_by_year.loc[compare_year]
    rows.append({
        "total/day of week": "total",
        "percent change": (compare_total / base_total - 1) * 100,
        "total ridership change": compare_total - base_total,
    })

    for day in day_of_week_order:
        base_value = jan_sep_by_day_year.loc[(base_year, day)]
        compare_value = jan_sep_by_day_year.loc[(compare_year, day)]
        rows.append({
            "total/day of week": day,
            "percent change": (compare_value / base_value - 1) * 100,
            "total ridership change": compare_value - base_value,
        })

    return pd.DataFrame(rows).sort_values(
        "percent change", key=abs, ascending=False
    )

table_2019_2026_day_of_week = make_day_of_week_comparison_table(2019, 2026)
table_2025_2026_day_of_week = make_day_of_week_comparison_table(2025, 2026)

In [13]:
def plot_day_of_week_changes(table, comparison):
    chart_data = table.loc[
        table["total/day of week"] != "total"
    ].copy()
    chart_data["direction"] = chart_data["percent change"].apply(
        lambda value: "Increase" if value >= 0 else "Decrease"
    )
    chart_data = chart_data.sort_values("percent change")

    fig = px.bar(
        chart_data,
        x="percent change",
        y="total/day of week",
        orientation="h",
        color="direction",
        color_discrete_map={"Increase": "#00cc96", "Decrease": "#EF553B"},
        hover_data={"total ridership change": ":,.0f"},
        title=f"Ridership Change by Day of Week: {comparison} (%)",
        labels={
            "percent change": "Change (%)",
            "total/day of week": "Day of Week",
            "total ridership change": "Ridership change",
        },
        category_orders={"total/day of week": day_of_week_order},
    )
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(showlegend=False)
    fig.show()


plot_day_of_week_changes(table_2019_2026_day_of_week, "2019–2026")
plot_day_of_week_changes(table_2025_2026_day_of_week, "2025–2026")

#### Takeaway

Looking at the ridership changes by day of week further illustrates that weekday ridership has decreased the most from 2019 to 2026. From 2025 to 2026, ridership has increased the most on Wednesdays, with a small decrease on Mondays. 

# Geographic Changes

In [14]:
combined_ridership_station = pd.concat(
    [
        ridership_2017_2019_station,
        ridership_2020_2024_station,
        ridership_2025_2026_station,
    ],
    ignore_index=True,
)

combined_ridership_station["date"] = pd.to_datetime(
    combined_ridership_station["date"]
)
combined_ridership_station = combined_ridership_station.sort_values(
    ["date", "station_complex"]
).reset_index(drop=True)

In [15]:
jan_sep_by_station_year = (
    combined_ridership_station.loc[
        combined_ridership_station["date"].dt.month.le(9)
    ]
    .assign(year=lambda df: df["date"].dt.year.astype(str))
    .groupby(["year", "station_complex_id"])["total_ridership"]
    .sum()
)

station_formatting = {
    "percent change": "{:+.2f}%",
    "total ridership change": "{:+,.0f}",
}

In [16]:
def make_station_comparison_table(base_year, compare_year):
    result = pd.concat(
        [
            jan_sep_by_station_year.xs(str(base_year), level="year")
            .rename("base_year_ridership"),
            jan_sep_by_station_year.xs(str(compare_year), level="year")
            .rename("compare_year_ridership"),
        ],
        axis=1,
    )

    for year in [base_year, compare_year]:
        # Use the latest observed name within January–September of each year.
        names = (
            combined_ridership_station.loc[
                combined_ridership_station["date"].dt.year.eq(year)
                & combined_ridership_station["date"].dt.month.le(9)
            ]
            .sort_values("date")
            .drop_duplicates("station_complex_id", keep="last")
            .set_index("station_complex_id")["station_complex"]
            .rename(f"station_complex_{year}")
        )
        result = result.join(names)

    result["percent change"] = (
        result["compare_year_ridership"]
        .div(result["base_year_ridership"].replace(0, float("nan")))
        .sub(1)
        .mul(100)
    )
    result["total ridership change"] = (
        result["compare_year_ridership"] - result["base_year_ridership"]
    )

    return (
        result.reset_index()[
            [
                "station_complex_id",
                f"station_complex_{base_year}",
                f"station_complex_{compare_year}",
                "base_year_ridership",
                "compare_year_ridership",
                "percent change",
                "total ridership change",
            ]
        ]
        .sort_values("percent change", key=abs, ascending=False)
        .reset_index(drop=True)
    )


table_2019_2026_station_with_names = make_station_comparison_table(2019, 2026)
table_2025_2026_station_with_names = make_station_comparison_table(2025, 2026)

### Borough

In [17]:
# Compare January–September totals for consistent year-to-year coverage.
ridership_station_with_details = combined_ridership_station.merge(
    station_complexes,
    left_on="station_complex_id",
    right_on="complex_id",
    how="left",
)

ridership_station_with_details.loc[
    ridership_station_with_details["station_complex_id"].isin(["TRAM1", "TRAM2"]),
    "borough",
] = "M"

borough_year_totals = (
    ridership_station_with_details.loc[
        ridership_station_with_details["date"].dt.month.le(9)
    ]
    .assign(
        year=lambda df: df["date"].dt.year,
        borough=lambda df: df["borough"].fillna("Unknown"),
    )
    .groupby(["borough", "year"])["total_ridership"]
    .sum(min_count=1)
    .unstack("year")
)

def make_borough_comparison(base_year, compare_year):
    result = borough_year_totals[[base_year, compare_year]].copy()
    result.columns = [
        f"ridership_{base_year}",
        f"ridership_{compare_year}",
    ]
    result["total ridership change"] = (
        result.iloc[:, 1] - result.iloc[:, 0]
    )
    result["percent change"] = (
        result["total ridership change"]
        .div(result.iloc[:, 0].replace(0, float("nan")))
        .mul(100)
    )
    return result.reset_index()

table_2019_2025_borough = make_borough_comparison(2019, 2025)
table_2025_2026_borough = make_borough_comparison(2025, 2026)

In [18]:
def plot_borough_changes(table, comparison):
    chart_data = table.assign(
        direction=lambda df: df["percent change"].apply(
            lambda value: "Increase" if value >= 0 else "Decrease"
        )
    ).sort_values("borough", ascending=True)

    borough_order = chart_data["borough"].tolist()

    borough_fig = px.bar(
        chart_data,
        x="percent change",
        y="borough",
        orientation="h",
        color="direction",
        color_discrete_map={"Increase": "#00cc96", "Decrease": "#EF553B"},
        hover_data={"total ridership change": ":,.0f"},
        title=f"Ridership Change by Borough: {comparison} (%)",
        labels={
            "percent change": "Change (%)",
            "borough": "Borough",
            "total ridership change": "Ridership change",
        },
        category_orders={"borough": borough_order},
    )

    borough_fig.add_vline(x=0, line_dash="dash", line_color="gray")
    borough_fig.update_layout(
        showlegend=False,
        yaxis={"categoryorder": "array", "categoryarray": borough_order, "autorange": "reversed"},
    )
    borough_fig.show()


plot_borough_changes(table_2019_2025_borough, "2019–2025")
plot_borough_changes(table_2025_2026_borough, "2025–2026")

#### Takeaway

Between 2019 and 2026, total ridership has decreased most drastically in Staten Island (minus 46.1%) and the Bronx (minus 35.5%). From 2025 to 2026, all boroughs has increase in total ridership, except the Bronx, where ridership decreased by 1.3%. 

In [19]:
complex_flags = station_complexes[["complex_id", "is_complex"]].assign(
    complex_id=lambda df: df["complex_id"].astype(str),
    is_complex=lambda df: df["is_complex"].astype(str).str.lower().eq("true"),
)

def filter_complex_stations(table):
    return (
        table.assign(station_complex_id=lambda df: df["station_complex_id"].astype(str))
        .merge(
            complex_flags,
            left_on="station_complex_id",
            right_on="complex_id",
            how="left",
            validate="many_to_one",
        )
        .loc[lambda df: df["is_complex"].eq(True)]
        .drop(columns=["complex_id", "is_complex"])
        .reset_index(drop=True)
    )

def top_changes(table, n=10):
    return (
        table.sort_values("percent change", key=abs, ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

table_2025_2026_station_complex_only = filter_complex_stations(table_2025_2026_station_with_names)
table_2019_2026_station_complex_only = filter_complex_stations(table_2019_2026_station_with_names)

top10_2025_2026_station_complex_only = top_changes(table_2025_2026_station_complex_only)
top10_2019_2026_station_complex_only = top_changes(table_2019_2026_station_complex_only)

top10_2025_2026_station_all = top_changes(table_2025_2026_station_with_names)
top10_2019_2026_station_all = top_changes(table_2019_2026_station_with_names)

In [20]:
station_change_scaled_maps = {}

def create_comparison_map(comparison_label, comparison_table):
    change_map_data = comparison_table.assign(
            station_complex_id=lambda df: df["station_complex_id"].astype(str)
        ).merge(
            station_complexes[["complex_id", "is_complex", "cbd", "borough", "latitude", "longitude"]].assign(
                complex_id=lambda df: df["complex_id"].astype(str)
            ),
            left_on="station_complex_id",
            right_on="complex_id",
            how="left",
            validate="many_to_one",
        )
    
    for field in ["latitude", "longitude", "percent change"]:
        change_map_data[field] = pd.to_numeric(
            change_map_data[field], errors="coerce"
        )

    change_map_data = change_map_data.replace(
        [float("inf"), -float("inf")], float("nan")
    ).dropna(subset=["latitude", "longitude", "percent change"])

    change_map_data["absolute_change"] = change_map_data["percent change"].abs()
    change_color_limit = max(
        change_map_data["absolute_change"].quantile(0.90), 1
    )

    change_map_fig = px.scatter_map(
        change_map_data,
        lat="latitude",
        lon="longitude",
        color="percent change",
        size="absolute_change",
        size_max=30,
        color_continuous_scale=[
            (0, "#ef4444"),
            (0.5, "#e5e7eb"),
            (1, "#4e22c5"),
        ],
        range_color=[-change_color_limit, change_color_limit],
        hover_name="station_complex_2026",
        hover_data={
            "station_complex_id": True,
            "percent change": ":.2f",
            "total ridership change": ":,.0f",
            "base_year_ridership": ":,.0f",
            "compare_year_ridership": ":,.0f",
            "absolute_change": False,
            "latitude": False,
            "longitude": False,
        },
        labels={"percent change": "Change (%)"},
        map_style="carto-darkmatter",
        template="plotly_dark",
        center={"lat": 40.73, "lon": -73.94},
        zoom=9.5,
        height=750,
        title=f"Station Ridership Change: {comparison_label} (January–September)",
    )
    change_map_fig.update_traces(marker={"opacity": 0.85, "sizemin": 3})
    station_change_scaled_maps[comparison_label] = change_map_fig
    change_map_fig.show()

for comparison_label, comparison_table in [
    ("2019–2026", table_2019_2026_station_with_names),
    ("2025–2026", table_2025_2026_station_with_names),
]:
    create_comparison_map(comparison_label, comparison_table)

The above two maps show overall total ridership changes per station, with larger changes represented by size of the circular marker, positive changes in blue, and negative changes in red. A quick glace shows that between 2019 and 2026, total ridership broadly decreased across the system, with more drastic decreases in the outer boroughs and the central business district. Between 2025 and 2026, total ridership broadly increased, mirroring the decreases seen between 2019 and 2026, with more dramatic increases in the outer boroughs and the central business district. Let's take a look more specifically at the top 10 biggest ridership changes across 2019-2026 and 2025-2026, as well as the top 10 biggest ridership changes within specifically subway station complexes.

In [ ]:
station_table_formatting = {
    **station_formatting,
    "base_year_ridership": "{:,.0f}",
    "compare_year_ridership": "{:,.0f}",
}

create_comparison_map("Top 10 changes, all stations: 2019–2026", top10_2019_2026_station_all)
display(top10_2019_2026_station_all.style.format(station_table_formatting, na_rep="N/A"))

,station_complex_id,station_complex_2019,station_complex_2026,base_year_ridership,compare_year_ridership,percent change,total ridership change
0,2,"Astoria Blvd (N,W)","Astoria Blvd (N,W)","664,595","1,940,133",+191.93%,"+1,275,538"
1,81,111 St (J),111 St (J),"188,081","408,236",+117.05%,"+220,155"
2,196,Aqueduct Racetrack (A),Aqueduct Racetrack (A),"467,812","148,227",-68.31%,"-319,585"
3,455,69 St (7),69 St (7),"1,142,010","490,735",-57.03%,"-651,275"
4,384,Burnside Av (4),Burnside Av (4),"2,312,033","1,183,618",-48.81%,"-1,128,415"
5,419,"225 St (2,5)","225 St (2,5)","887,566","457,776",-48.42%,"-429,790"
6,210,Norwood-205 St (D),Norwood-205 St (D),"1,969,619","1,023,233",-48.05%,"-946,386"
7,457,52 St (7),52 St (7),"1,399,519","727,746",-48.00%,"-671,773"
8,213,"Fordham Rd (B,D)","Fordham Rd (B,D)","2,728,914","1,419,062",-48.00%,"-1,309,852"
9,212,"Kingsbridge Rd (B,D)","Kingsbridge Rd (B,D)","1,837,579","957,008",-47.92%,"-880,571"


When looking at just the Top 10 biggest changes across all stations from 2019-2026, you can see even more clearly that the largest changes were concentrated in the outer boroughs. Interestingly, two of the largest changes were increases in ridership at Astoria Blvd and 111 St. Astoria Blvd's increase in ridership may be due to the fact the station was fully closed for renovations for 8 months.

In [30]:
create_comparison_map("Top 10 changes, all stations: 2025–2026", top10_2025_2026_station_all)
display(top10_2025_2026_station_all.style.format(station_table_formatting, na_rep="N/A"))

,station_complex_id,station_complex_2025,station_complex_2026,base_year_ridership,compare_year_ridership,percent change,total ridership change
0,207,Beach 36 St (A),Beach 36 St (A),"65,439","143,773",+119.71%,"+78,334"
1,86,Cypress Hills (J),Cypress Hills (J),"102,097","206,397",+102.16%,"+104,300"
2,208,Beach 25 St (A),Beach 25 St (A),"120,932","236,927",+95.92%,"+115,995"
3,205,Beach 60 St (A),Beach 60 St (A),"146,834","284,138",+93.51%,"+137,304"
4,206,Beach 44 St (A),Beach 44 St (A),"61,209","111,537",+82.22%,"+50,328"
5,199,"Broad Channel (A,S)","Broad Channel (A,S)","27,460","49,962",+81.94%,"+22,502"
6,209,Far Rockaway-Mott Av (A),Far Rockaway-Mott Av (A),"476,832","846,707",+77.57%,"+369,875"
7,204,Beach 67 St (A),Beach 67 St (A),"232,824","396,429",+70.27%,"+163,605"
8,203,"Rockaway Park-Beach 116 St (A,S)","Rockaway Park-Beach 116 St (A,S)","92,732","155,943",+68.17%,"+63,211"
9,84,85 St-Forest Pkwy (J),85 St-Forest Pkwy (J),"371,525","590,500",+58.94%,"+218,975"


From 2025 to 2026, the largest changes in total ridership were all increases in total ridership. 8 of the 10 largest increases in ridership were in the Rockaways, which may be due to a combination of an influx in summer travel to the Rockaways and increases in both A and Shuttle service to the Rockaways during the summertime.

In [31]:
create_comparison_map("Top 10 changes, station complexes only: 2019–2026", top10_2019_2026_station_complex_only)
display(top10_2019_2026_station_complex_only.style.format(station_table_formatting, na_rep="N/A"))

,station_complex_id,station_complex_2019,station_complex_2026,base_year_ridership,compare_year_ridership,percent change,total ridership change
0,620,"Court St (R)/Borough Hall (2,3,4,5)","Borough Hall/Court St (2,3,4,5,R)","7,748,075","5,022,557",-35.18%,"-2,725,518"
1,603,"149 St-Grand Concourse (2,4,5)","149 St-Grand Concourse (2,4,5)","2,914,054","1,894,990",-34.97%,"-1,019,064"
2,635,"South Ferry (1)/Whitehall St (R,W)","Whitehall St-South Ferry (1,R,W)","7,747,086","5,262,846",-32.07%,"-2,484,240"
3,607,"34 St-Herald Sq (B,D,F,M,N,Q,R,W)","34 St-Herald Sq (B,D,F,M,N,Q,R,W)","28,882,383","19,903,775",-31.09%,"-8,978,608"
4,636,"Jay St-MetroTech (A,C,F,R)","Jay St-MetroTech (A,C,F,R)","9,256,568","6,403,689",-30.82%,"-2,852,879"
5,622,"Brooklyn Bridge-City Hall (4,5,6)/Chambers St (J,Z)","Brooklyn Bridge-City Hall/Chambers St (4,5,6,J,Z)","6,793,078","4,735,698",-30.29%,"-2,057,380"
6,624,"Chambers St (A,C)/WTC (E)/Park Pl (2,3)/Cortlandt (R,W)","Chambers St/WTC/Park Place/Cortlandt St (2,3,A,C,E,R,W)","15,352,605","10,909,714",-28.94%,"-4,442,891"
7,613,"Lexington Av (N,R,W)/59 St (4,5,6)","Lexington Av/59 St (4,5,6,N,R,W)","12,371,020","8,886,755",-28.16%,"-3,484,265"
8,627,"Franklin Av (C,S)","Franklin Av (C,S)","1,486,919","1,076,827",-27.58%,"-410,092"
9,628,"Fulton St (A,C,J,Z,2,3,4,5)","Fulton St (2,3,4,5,A,C,J,Z)","20,574,729","15,194,582",-26.15%,"-5,380,147"


When looking at the Top 10 changes in ridership at subway station complexes between 2021 to 2026, 6 of the top 10 stations cluster around the Financial District and Downtown Brooklyn.

In [32]:
create_comparison_map("Top 10 changes, station complexes only: 2025–2026", top10_2025_2026_station_complex_only)
display(top10_2025_2026_station_complex_only.style.format(station_table_formatting, na_rep="N/A"))

,station_complex_id,station_complex_2025,station_complex_2026,base_year_ridership,compare_year_ridership,percent change,total ridership change
0,621,"Broadway Junction (A,C,J,Z,L)","Broadway Junction (A,C,J,Z,L)","1,461,068","1,664,520",+13.92%,"+203,452"
1,612,"Lexington Av/51-53 Sts (6,E,F)","Lexington Av/51-53 Sts (6,E,F)","9,840,158","10,987,923",+11.66%,"+1,147,765"
2,606,"Court Sq-23 St (7,E,F,G)","Court Sq-23 St (7,E,F,G)","4,770,239","5,260,540",+10.28%,"+490,301"
3,625,"Delancey St-Essex St (F,M,J,Z)","Delancey St-Essex St (F,M,J,Z)","5,781,831","6,258,790",+8.25%,"+476,959"
4,601,"14 St/6 Av (1,2,3,L,F,M)","14 St/6 Av (1,2,3,L,F,M)","8,423,332","9,052,676",+7.47%,"+629,344"
5,609,"42 St-Bryant Pk/5 Av (7,B,D,F,M)","42 St-Bryant Pk/5 Av (7,B,D,F,M)","9,421,587","10,060,032",+6.78%,"+638,445"
6,604,"161 St-Yankee Stadium (4,B,D)","161 St-Yankee Stadium (4,B,D)","5,093,156","4,757,358",-6.59%,"-335,798"
7,619,"Broadway-Lafayette St/Bleecker St (6,B,D,F,M)","Broadway-Lafayette St/Bleecker St (6,B,D,F,M)","7,945,244","8,392,868",+5.63%,"+447,624"
8,608,"4 Av-9 St (F,G,R)","4 Av-9 St (F,G,R)","2,339,972","2,468,389",+5.49%,"+128,417"
9,627,"Franklin Av (C,S)","Franklin Av (C,S)","1,028,346","1,076,827",+4.71%,"+48,481"


When looking at the Top 10 changes in ridership at subway station complexes between 2025 to 2026, the largest increase in ridership was at Broadway Junction.  

### Origin/Destination Analysis - Central Business District (CBD)

In [25]:
combined_od_september = pd.concat([od_2021_september, od_2025_september, od_2026_september])

In [26]:
station_details_for_september = station_complexes.copy()
station_details_for_september["complex_id"] = (
    station_details_for_september["complex_id"].astype("string")
)

september_od = combined_od_september.copy()
for column in ["origin_station_complex_id", "destination_station_complex_id"]:
    september_od[column] = september_od[column].astype("string")

origin_details = station_details_for_september.rename(
    columns={
        column: f"origin_{column}"
        for column in station_details_for_september.columns
        if column != "complex_id"
    } | {"complex_id": "origin_station_complex_id"}
)

destination_details = station_details_for_september.rename(
    columns={
        column: f"destination_{column}"
        for column in station_details_for_september.columns
        if column != "complex_id"
    } | {"complex_id": "destination_station_complex_id"}
)

combined_od_september_with_station_details = (
    september_od
    .merge(
        origin_details,
        on="origin_station_complex_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        destination_details,
        on="destination_station_complex_id",
        how="left",
        validate="many_to_one",
    )
)

In [27]:
cbd_origin_map_years = [2021, 2025, 2026]
available_years = set(
    pd.to_numeric(combined_od_september_with_station_details["year"], errors="coerce")
    .dropna()
    .astype(int)
)

cbd_destination_trips = combined_od_september_with_station_details.assign(
    year=lambda df: pd.to_numeric(df["year"], errors="coerce"),
    avg_daily_ridership=lambda df: pd.to_numeric(
        df["avg_daily_ridership"], errors="coerce"
    ),
    origin_latitude=lambda df: pd.to_numeric(df["origin_latitude"], errors="coerce"),
    origin_longitude=lambda df: pd.to_numeric(df["origin_longitude"], errors="coerce"),
)
cbd_destination_trips = cbd_destination_trips.loc[
    cbd_destination_trips["destination_cbd"].astype("string").str.lower().eq("true")
    & cbd_destination_trips["year"].isin(cbd_origin_map_years)
]

cbd_origin_by_station = (
    cbd_destination_trips
    .groupby(
        [
            "year",
            "origin_station_complex_id",
            "origin_display_name",
            "origin_borough",
            "origin_latitude",
            "origin_longitude",
        ],
        as_index=False,
        dropna=False,
    )["avg_daily_ridership"]
    .sum()
    .rename(columns={"avg_daily_ridership": "cbd_bound_avg_daily_ridership"})
    .dropna(subset=["origin_latitude", "origin_longitude"])
)
cbd_origin_by_station["year"] = cbd_origin_by_station["year"].astype(int).astype(str)

cbd_origin_year_totals = cbd_origin_by_station.pivot_table(
    index="origin_station_complex_id",
    columns="year",
    values="cbd_bound_avg_daily_ridership",
    aggfunc="sum",
)

cbd_origin_metadata = cbd_origin_by_station.drop_duplicates(
    "origin_station_complex_id", keep="last"
)[[
    "origin_station_complex_id",
    "origin_display_name",
    "origin_borough",
    "origin_latitude",
    "origin_longitude",
]]

cbd_origin_change_maps = {}

for cbd_base_year in [2021]:
    cbd_change_data = cbd_origin_year_totals[
        [str(cbd_base_year), "2026"]
    ].copy()
    cbd_change_data.columns = ["base_ridership", "ridership_2026"]
    cbd_change_data["percent_change"] = (
        cbd_change_data["ridership_2026"]
        .div(cbd_change_data["base_ridership"].replace(0, float("nan")))
        .sub(1)
        .mul(100)
    )
    cbd_change_data = (
        cbd_change_data.reset_index()
        .merge(
            cbd_origin_metadata,
            on="origin_station_complex_id",
            validate="one_to_one",
        )
        .replace([float("inf"), -float("inf")], float("nan"))
        .dropna(subset=["percent_change", "origin_latitude", "origin_longitude"])
    )

    cbd_color_limit = max(
        cbd_change_data["percent_change"].abs().quantile(0.85), 1
    )
    cbd_comparison_label = f"{cbd_base_year}–2026"

    cbd_change_fig = px.scatter_map(
        cbd_change_data,
        lat="origin_latitude",
        lon="origin_longitude",
        color="percent_change",
        color_continuous_scale=["#ef4444", "#e5e7eb", "#22c55e"],
        range_color=[-cbd_color_limit, cbd_color_limit],
        hover_name="origin_display_name",
        hover_data={
            "origin_borough": True,
            "percent_change": ":+.2f",
            "base_ridership": ":,.0f",
            "ridership_2026": ":,.0f",
            "origin_latitude": False,
            "origin_longitude": False,
        },
        labels={
            "percent_change": "Change (%)",
            "base_ridership": f"{cbd_base_year} ridership",
            "ridership_2026": "2026 ridership",
        },
        map_style="carto-darkmatter",
        template="plotly_dark",
        center={"lat": 40.73, "lon": -73.94},
        zoom=9.5,
        height=750,
        title=f"CBD-Bound Ridership Change by Origin: {cbd_comparison_label}",
    )
    cbd_change_fig.update_traces(marker={"size": 10, "opacity": 0.85})
    cbd_origin_change_maps[cbd_comparison_label] = cbd_change_fig
    cbd_change_fig.show()

In [33]:
cbd_totals = (
    cbd_destination_trips.groupby("year")["avg_daily_ridership"]
    .sum()
)

cbd_internal_trips = combined_od_september_with_station_details.assign(
    year=lambda df: pd.to_numeric(df["year"], errors="coerce"),
    avg_daily_ridership=lambda df: pd.to_numeric(
        df["avg_daily_ridership"], errors="coerce"
    ),
)

cbd_internal_trips = cbd_internal_trips.loc[
    cbd_internal_trips["origin_cbd"].astype("string").str.lower().eq("true")
    & cbd_internal_trips["destination_cbd"].astype("string").str.lower().eq("true")
]

internal_cbd_totals = cbd_internal_trips.groupby("year")[
    "avg_daily_ridership"
].sum()

destination_change = (
    cbd_totals.loc[2026] / cbd_totals.loc[2021] - 1
) * 100

internal_change = (
    internal_cbd_totals.loc[2026] / internal_cbd_totals.loc[2021] - 1
) * 100

#### Takeaway

When comparing trips to the Central Business District from September 2021 to Septemeber 2026, you can see from the map that there are broad increases from origins across the system, but there seems to be a concentration of increases of trips from within the CBD. 

Trips with a destination of the CBD increased by 54.31%, while trips within the CBD increased by +89.65%.

## Key Findings

**Weekdays**: Total ridership has still not recovered to 2019 ridership numbers post-COVID. Weekdays show the most drastic differences between 2019 and 2026, with a 24.47% decrease in ridership on weekdays in the reporting period between 2019 and 2026. Much of this difference can be attributed to less ridership during the typical workday, with AM Rush Hours showing the largest drop in ridership (-30.9%). 

- *Futher analysis recommended*: If ridership has more broadly recovered to 2019 numbers during off-peak hours, like during late nights and weekends, NYCT should consider even more off-peak service increases.   

**The Bronx**: Between 2019 and 2026, total ridership in the Bronx decreased by the 2nd most of all the boroughs (after Staten Island) by 35.5%. From 2025 to 2026, all boroughs has increase in total ridership, except the Bronx, where ridership decreased by 1.3%. 

- *Further analysis recommended*: Investigate the cause of continued low ridership in the Bronx; was the decrease in ridership in the Bronx due to population/socioeconomoic patterns? Or is the decrease in ridership due to less service scheduled/less service due to incidents or disruptive Track Outages? 

**The Rockaways**: From 2025 to 2026, the largest changes in total ridership were all increases in total ridership. 8 of the 10 largest increases in ridership were in the Rockaways.

- *Further analysis recommended*: I believe there have been a few seasonal service changes to the Rockaway lines within the past few years. It would be great to investigate how these seasonal service changes have impacted overall visitor numbers and economic growth in the area. NYCT could take these findings and institute more seasonal service changes to other parts of the system to benefit other key economic zones. 

## Appendix: Data Preparation, Assumptions, Limitations

### Data Sources: 
- [MTA Subway Hourly Ridership: 2017-2019](https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-2017-2019/t69i-h2me/about_data)
- [MTA Subway Hourly Ridership: 2020-2024](https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-2020-2024/wujg-7c2s/about_data)
- [MTA Subway Hourly Ridership: Beginning 2025](https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-Beginning-2025/5wq4-mkjj/about_data)
- [MTA Subway Stations and Complexes](https://data.ny.gov/Transportation/MTA-Subway-Stations-and-Complexes/5f5g-n3cz/about_data)
- [MTA Subway Origin-Destination Ridership Estimate: 2021](https://data.ny.gov/Transportation/MTA-Subway-Origin-Destination-Ridership-Estimate-2/rapa-97zv/about_data) 
- [MTA Subway Origin-Destination Ridership Estimate: 2025](https://data.ny.gov/Transportation/MTA-Subway-Origin-Destination-Ridership-Estimate-2/y2qv-fytt/about_data)
- [MTA Subway Origin-Destination Ridership Estimate: Beginning 2026](https://data.ny.gov/Transportation/MTA-Subway-Origin-Destination-Ridership-Estimate-B/28vm-gjqr/about_data)

I extracted aggregated or filtered datasets directly from the API endpoints in the `ridership_data_sources.ipynb` notebook. 

Generic subway time periods definitions from [New York City Subway services: Time Periods](https://en.wikipedia.org/wiki/List_of_New_York_City_Subway_services#Time_periods)

### Assumptions
I used median when calculating the average of the weekday/weekend ridership. This is because the weekday ridership data had enough outliers on the lower range of the data to pull down the mean. 

### Limitations 
Since 2026 is not over yet, analysis involving total ridership in 2026 must be compared to a similar reporting period for other years. I chose January - September so we had a full months of data to compare. 

For "average ridership" numbers, I still calculated average based on all the data available for each year and compared this to each other, assuming that 9+ months of data was a reliable amount to come up with comparable average ridership numbers across years. 

Origin-Destination data was only available for 2021 and later. Due to the Origin-Destination data being provided as estimated averages already, it is more difficult to aggregate this data to assume yearly or monthly origin/destination data. The origin/destination dataset is also quite large due to the sheer number of origin/destination combinations. Because of this, for this exercise I compared 1 month of data across 2 years. 

In [29]:
distribution_data = jan_sep_daily.loc[
    jan_sep_daily["year"].isin(["2019", "2025", "2026"])
].copy()

fig = px.histogram(
    distribution_data,
    x="total_ridership",
    color="year",
    facet_col="day_type",
    barmode="overlay",
    histnorm="probability density",
    opacity=0.55,
    marginal="box",
    title="Daily Ridership Distribution, January–September",
    labels={
        "total_ridership": "Daily Ridership",
        "year": "Year",
        "day_type": "Day Type",
    },
)
fig.show()

def summarize_distribution(group):
    values = group["total_ridership"]
    return pd.Series({
        "days": values.count(),
        "mean": values.mean(),
        "median": values.median(),
        "std_dev": values.std(),
        "skew": values.skew(),
        "25th percentile": values.quantile(0.25),
        "75th percentile": values.quantile(0.75),
        "min": values.min(),
        "max": values.max(),
    })

distribution_summary = (
    distribution_data.groupby(["year", "day_type"])
    .apply(summarize_distribution, include_groups=False)
    .reset_index()
)

display(distribution_summary.style.format({
    "mean": "{:,.0f}",
    "median": "{:,.0f}",
    "std_dev": "{:,.0f}",
    "skew": "{:.2f}",
    "25th percentile": "{:,.0f}",
    "75th percentile": "{:,.0f}",
    "min": "{:,.0f}",
    "max": "{:,.0f}",
}))

,year,day_type,days,mean,median,std_dev,skew,25th percentile,75th percentile,min,max
0,2019,Weekday,195.000000,"5,409,943","5,515,120","607,296",-3.44,"5,283,688","5,731,450","2,157,012","6,075,964"
1,2019,Weekend,78.000000,"2,688,545","2,607,619","391,222",0.04,"2,404,118","3,010,593","1,884,734","3,369,995"
2,2025,Weekday,195.000000,"3,986,184","4,009,721","423,341",-2.21,"3,806,688","4,262,192","1,813,725","4,571,811"
3,2025,Weekend,78.000000,"2,405,953","2,363,680","336,450",0.03,"2,139,096","2,692,064","1,818,179","3,027,047"
4,2026,Weekday,195.000000,"4,086,162","4,145,917","523,714",-2.69,"3,933,872","4,442,706","861,364","4,787,759"
5,2026,Weekend,78.000000,"2,465,992","2,416,040","424,666",-0.63,"2,209,410","2,836,271","850,416","3,297,447"
